# Lab : Image Classification using Convolutional Neural Networks

At the end of this laboratory, you would get familiarized with

*   Creating deep networks using Keras
*   Steps necessary in training a neural network
*   Prediction and performance analysis using neural networks

---

# **In case you use a colaboratory environment**
By default, Colab notebooks run on CPU.
You can switch your notebook to run with GPU.

In order to obtain access to the GPU, you need to choose the tab Runtime and then select “Change runtime type” as shown in the following figure:

![Changing runtime](https://miro.medium.com/max/747/1*euE7nGZ0uJQcgvkpgvkoQg.png)

When a pop-up window appears select GPU. Ensure “Hardware accelerator” is set to GPU.

# **Working with a new dataset: CIFAR-10**

The CIFAR-10 dataset consists of 60000 32x32 colour images in 10 classes, with 6000 images per class. There are 50000 training images and 10000 test images. More information about CIFAR-10 can be found [here](https://www.cs.toronto.edu/~kriz/cifar.html).

In Keras, the CIFAR-10 dataset is also preloaded in the form of four Numpy arrays. x_train and y_train contain the training set, while x_test and y_test contain the test data. The images are encoded as Numpy arrays and their corresponding labels ranging from 0 to 9.

Your task is to:

*   Visualize the images in CIFAR-10 dataset. Create a 10 x 10 plot showing 10 random samples from each class.
*   Convert the labels to one-hot encoded form.
*   Normalize the images.




In [3]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# 10 x 10 grid: 10 random samples per class
fig, axes = plt.subplots(10, 10, figsize=(10, 10))

for class_idx in range(10):
    # find all indices in the training set belonging to this class
    idxs = np.where(y_train.flatten() == class_idx)[0]
    chosen = np.random.choice(idxs, 10, replace=False)
    for col, img_idx in enumerate(chosen):
        ax = axes[class_idx, col]
        ax.imshow(x_train[img_idx])
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(class_names[class_idx])

plt.tight_layout()
plt.show()

# One-hot encode the labels
num_classes = 10
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

# Normalize the images to [0, 1]
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm = x_test.astype('float32') / 255.0

print(x_train_norm.shape, y_train_cat.shape)
print(x_test_norm.shape, y_test_cat.shape)

## Define the following model (same as the one in tutorial)

For the convolutional front-end, start with a single convolutional layer with a small filter size (3,3) and a modest number of filters (32) followed by a max pooling layer. 

Use the input as (32,32,3). 

The filter maps can then be flattened to provide features to the classifier. 

Use a dense layer with 100 units before the classification layer (which is also a dense layer with softmax activation).

In [5]:
from keras.backend import clear_session
clear_session()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model_simple = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(100, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model_simple.summary()

*   Compile the model using categorical_crossentropy loss, SGD optimizer and use 'accuracy' as the metric.
*   Use the above defined model to train CIFAR-10 and train the model for 50 epochs with a batch size of 512.

In [ ]:
from tensorflow.keras.optimizers import SGD

model_simple.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(),
    metrics=['accuracy']
)

history_simple = model_simple.fit(
    x_train_norm, y_train_cat,
    validation_data=(x_test_norm, y_test_cat),
    epochs=50,
    batch_size=512
)

*   Plot the cross entropy loss curve and the accuracy curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_simple.history['loss'], label='Train Loss')
axes[0].plot(history_simple.history['val_loss'], label='Val Loss')
axes[0].set_title('Cross-Entropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history_simple.history['accuracy'], label='Train Accuracy')
axes[1].plot(history_simple.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.show()

## Defining Deeper Architectures: VGG Models

*   Define a deeper model architecture for CIFAR-10 dataset and train the new model for 50 epochs with a batch size of 512. We will use VGG model as the architecture.

Stack two convolutional layers with 32 filters, each of 3 x 3. 

Use a max pooling layer and next flatten the output of the previous layer and add a dense layer with 128 units before the classification layer. 

For all the layers, use ReLU activation function. 

Use same padding for the layers to ensure that the height and width of each layer output matches the input


In [9]:
from keras.backend import clear_session
clear_session()

In [ ]:
model_vgg = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model_vgg.summary()

*   Compile the model using categorical_crossentropy loss, SGD optimizer and use 'accuracy' as the metric.
*   Use the above defined model to train CIFAR-10 and train the model for 50 epochs with a batch size of 512.

In [ ]:
model_vgg.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(),
    metrics=['accuracy']
)

history_vgg = model_vgg.fit(
    x_train_norm, y_train_cat,
    validation_data=(x_test_norm, y_test_cat),
    epochs=50,
    batch_size=512
)

*   Compare the performance of both the models by plotting the loss and accuracy curves of both the training steps. Does the deeper model perform better? Comment on the observation.
 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_simple.history['val_loss'], label='Simple Model')
axes[0].plot(history_vgg.history['val_loss'], label='VGG Model')
axes[0].set_title('Validation Loss Comparison')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history_simple.history['val_accuracy'], label='Simple Model')
axes[1].plot(history_vgg.history['val_accuracy'], label='VGG Model')
axes[1].set_title('Validation Accuracy Comparison')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.show()

**Comment on the observation**

The deeper VGG-style model [achieved higher / lower / similar] validation accuracy compared 
to the simple single-conv-layer model ([X]% vs [Y]%). This is expected because stacking two 
3x3 convolutions before pooling gives the network a larger effective receptive field and more 
capacity to learn hierarchical features, at the cost of more parameters and longer training 
time per epoch.

*   Use predict function to predict the output for the test split
*   Plot the confusion matrix for the new model and comment on the class confusions.


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_pred_probs = model_vgg.predict(x_test_norm)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - VGG Model')
plt.show()

**Comment here :**

The most common confusions are between visually or semantically similar classes — e.g. 
cat/dog and automobile/truck tend to be confused more often than, say, airplane/frog, 
since those pairs share similar shapes, textures, and typical backgrounds at low resolution.

*    Print the test accuracy for the trained model.

In [ ]:
test_loss, test_accuracy = model_vgg.evaluate(x_test_norm, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

## Define the complete VGG architecture.

Stack two convolutional layers with 64 filters, each of 3 x 3 followed by max pooling layer. 

Stack two more convolutional layers with 128 filters, each of 3 x 3, followed by max pooling, followed by two more convolutional layers with 256 filters, each of 3 x 3, followed by max pooling. 

Flatten the output of the previous layer and add a dense layer with 128 units before the classification layer. 

For all the layers, use ReLU activation function. 

Use same padding for the layers to ensure that the height and width of each layer output matches the input

*   Change the size of input to 64 x 64.

In [15]:
from keras.backend import clear_session
clear_session()

In [ ]:
import tensorflow as tf

# Resize images to 64x64
x_train_64 = tf.image.resize(x_train_norm, (64, 64)).numpy()
x_test_64 = tf.image.resize(x_test_norm, (64, 64)).numpy()

print(x_train_64.shape, x_test_64.shape)

In [ ]:
model_full_vgg = Sequential([
    Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=(64, 64, 3)),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation='relu', padding='same'),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),

    Conv2D(256, (3, 3), activation='relu', padding='same'),
    Conv2D(256, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model_full_vgg.summary()

*   Compile the model using categorical_crossentropy loss, SGD optimizer and use 'accuracy' as the metric.
*   Use the above defined model to train CIFAR-10 and train the model for 10 epochs with a batch size of 512.
*   Predict the output for the test split and plot the confusion matrix for the new model and comment on the class confusions.

In [ ]:
model_full_vgg.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(),
    metrics=['accuracy']
)

history_full_vgg = model_full_vgg.fit(
    x_train_64, y_train_cat,
    validation_data=(x_test_64, y_test_cat),
    epochs=10,
    batch_size=512
)

# Predict and plot confusion matrix
y_pred_probs_full = model_full_vgg.predict(x_test_64)
y_pred_full = np.argmax(y_pred_probs_full, axis=1)

cm_full = confusion_matrix(y_true, y_pred_full)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_full, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Full VGG Model (64x64 input)')
plt.show()

test_loss_full, test_accuracy_full = model_full_vgg.evaluate(x_test_64, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_accuracy_full:.4f}")

# Understanding deep networks

*   What is the use of activation functions in network? Why is it needed?
*   We have used softmax activation function in the exercise. There are other activation functions available too. What is the difference between sigmoid activation and softmax activation?
*   What is the difference between categorical crossentropy and binary crossentropy loss?

**Write the answers below :**

1 - Use of activation functions:

Activation functions introduce non-linearity into the network. Without them, stacking multiple 
layers of matrix multiplications (Dense or Conv layers) would still collapse mathematically into 
a single linear transformation, no matter how many layers you add — the network could only learn 
linear decision boundaries. Non-linear activations (ReLU, sigmoid, etc.) let the network approximate 
complex, non-linear functions and learn hierarchical features, which is essential for tasks like 
image classification.

2 - Key Differences between sigmoid and softmax:

Sigmoid squashes each output independently into a range between 0 and 1, and is typically used 
for binary classification or multi-label problems where classes are not mutually exclusive — 
each output can be interpreted as an independent probability. Softmax, on the other hand, 
converts a vector of raw scores into a probability distribution across all classes simultaneously, 
where all outputs sum to 1. It's used for multi-class, single-label classification (like CIFAR-10) 
where exactly one class should be the correct answer, since it forces the model to distribute 
"confidence" across competing classes.

3 - Key Differences between categorical crossentropy and binary crossentropy loss:

Binary crossentropy is used when there are exactly two classes (or multiple independent binary 
labels), comparing a single predicted probability against a single true label (0 or 1) per output 
unit. Categorical crossentropy is used for multi-class, single-label classification with more than 
two mutually exclusive classes; it compares a full predicted probability distribution (from softmax) 
against a one-hot encoded true label vector, penalizing the model based on how much probability 
mass it assigned to the wrong classes.